## Gradient Boosting Machines (GBM)

Gradient Boosting builds a prediction function $F(x)$ **additively** by starting from a single constant and then fitting many small trees to the “leftover” errors.  First you choose

$$
F_0(x) \;=\;\arg\min_{\gamma}\sum_{i=1}^n\ell\bigl(y_i,\gamma\bigr),
$$

i.e. the best constant predictor (mean of $y_i$ for squared‐error, log-odds for logistic loss).

<br>

<p align="center">
<img src="visualizations/GBM.png" width="600">
</p>


At each iteration $m=1,\dots,M$:

1. **Compute pseudo-residuals**

   $$
     r_{i,m}
     = -\left.\frac{\partial}{\partial F(x_i)}\ell\bigl(y_i,F(x_i)\bigr)\right|_{F=F_{m-1}},
   $$

   which generalizes the usual “residual” $y_i - \hat y_i$ to any differentiable loss.  For squared error, $r_{i,m}=y_i-F_{m-1}(x_i)$; for logistic, $r_{i,m}=y_i - p_{i,m-1}$ with $p=\sigma(F)$.

2. **Fit a small regression tree** $h_m(x)$ to $\{(x_i,r_{i,m})\}$.  This partitions the input space into regions $R_{j,m}$, $j=1,\dots,J$.

3. **Compute the per-leaf updates**

   $$
     \gamma_{j,m}
     = \arg\min_{\gamma}\sum_{x_i\in R_{j,m}}
       \ell\bigl(y_i,\,F_{m-1}(x_i)+\gamma\bigr),
   $$

   i.e. the optimal constant offset in each leaf.  In squared loss $\gamma_{j,m}$ is just the mean of the residuals in leaf $j$; in logistic loss it’s a Newton-step ratio $\sum(y_i-p)/\sum[p(1-p)]$.

4. **Update the ensemble**

   $$
     F_m(x)
     = F_{m-1}(x)
       + \eta\sum_{j=1}^J \gamma_{j,m}\,\mathbf1\{x\in R_{j,m}\},
   $$

   where $\eta\in(0,1]$ is the learning rate (shrinkage).

Repeat until $m=M$, yielding

$$
F_M(x) \;=\; F_0(x) + \sum_{m=1}^M \eta\,h_m(x).
$$

---

**Key intuitions**:

* This is **gradient descent in function-space**: each tree $h_m$ is a “search direction” (fit to negative gradients), and each $\gamma_{j,m}$ is the “step size” (optimal line search) in that region.
* You never build one huge tree; instead you take **many tiny steps**, which controls overfitting and makes tuning (tree depth, $\eta$, subsampling) more effective.
* **Pseudo-residuals** tell you locally which way to adjust your prediction to reduce loss most quickly.
* $\gamma$ appears first as the global starting constant $F_0$, and thereafter as the per-leaf offset that best corrects the current model in that region.

By iterating “direction = pseudo-residuals, step size = $\gamma$, add tree,” GBM converges to a strong learner even when each individual tree is very weak.


In [6]:
import numpy as np
from tqdm import tqdm
from cifar10.unpickle import get_all_data, get_test_data

from sklearn.tree import DecisionTreeRegressor

In [7]:
class GradientBoostingClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []  # list of lists: trees[m][k]
        self.K = None
        self.F0 = None

    def _softmax(self, F):
        e = np.exp(F - np.max(F, axis=1, keepdims=True))
        return e / np.sum(e, axis=1, keepdims=True)

    def fit(self, X, y):
        # X: (n_samples, n_features), y: (n_samples,)
        n, _ = X.shape
        self.K = np.max(y) + 1  # number of classes
        # One-hot encode y
        Y = np.eye(self.K)[y]

        # Initialize F0 as log odds
        pi = np.mean(Y, axis=0)
        self.F0 = np.log(pi + 1e-9)
        F = np.tile(self.F0, (n, 1))  # shape (n, K)

        for m in tqdm(range(self.n_estimators), desc="Training GBM"):
            trees_m = []
            P = self._softmax(F)
            for k in range(self.K):
                # pseudo-residuals
                r_k = Y[:, k] - P[:, k]
                tree = DecisionTreeRegressor(max_depth=self.max_depth)
                tree.fit(X, r_k)
                update = tree.predict(X)
                F[:, k] += self.learning_rate * update
                trees_m.append(tree)
            self.trees.append(trees_m)

    def predict_proba(self, X):
        n = X.shape[0]
        F = np.tile(self.F0, (n, 1))
        for trees_m in self.trees:
            for k, tree in enumerate(trees_m):
                F[:, k] += self.learning_rate * tree.predict(X)
        return self._softmax(F)

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

In [8]:
# 1) Load
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

y_train = np.array(y_train)
y_test = np.array(y_test)

n_samples, h, w, c = x_train.shape  # h=32, w=32, c=3
x_train = x_train.reshape(n_samples, h * w * c)  # flatten to (N, 3072)
x_test = x_test.reshape(x_test.shape[0], h * w * c)  # flatten to (N, 3072)

In [9]:
# Train GBM
gbm = GradientBoostingClassifier(n_estimators=20, learning_rate=0.5, max_depth=1)
gbm.fit(x_train, y_train)

# Evaluate
from sklearn.metrics import accuracy_score

y_pred = gbm.predict(x_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

Training GBM: 100%|██████████| 20/20 [18:19<00:00, 54.99s/it]


Test accuracy: 0.2822


TODO: Experiment with data preprocessing. Use dimensionality reduction or extract image descriptors through filtering